# Hermes — Regime Detection + Walk-Forward Backtest

**Scope**: Research prototype only — Hermes Quant Analyst workspace.  
**Goal**: Detect market regimes on VN30F data, run regime-aware signal generation, and walk-forward validate.

> ⚠️ This notebook is for research/hypothesis validation only. No production trading logic is implemented here.

In [ ]:
import sys, pathlib
ROOT = pathlib.Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from research.data.loader import load_ohlcv
from research.features.technical import add_returns, add_volatility, add_momentum
from research.features.microstructure import add_spread_proxy, add_amihud_illiquidity
from research.regime.hmm_regime import HMMRegimeDetector
from research.regime.volatility_regime import VolatilityRegime
from research.signals.mean_reversion import ZScoreMeanReversion
from research.signals.momentum import TimeSeriesMomentum
from research.backtest.engine import run_backtest
from research.backtest.walkforward import walk_forward_evaluate
from research.backtest.metrics import sharpe_ratio, max_drawdown

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
print('Imports OK')

## 1. Load & Prepare Data

In [ ]:
DATA_PATH = '../data/vn30f_1d.csv'

try:
    df = load_ohlcv(DATA_PATH)
except FileNotFoundError:
    np.random.seed(0)
    n = 600
    idx = pd.bdate_range('2021-01-01', periods=n, tz='Asia/Ho_Chi_Minh')
    # Synthetic regime-switching process
    regimes = np.repeat([0, 1, 2, 0, 1], [150, 100, 100, 150, 100])
    vols = np.where(regimes == 0, 5, np.where(regimes == 1, 12, 20))
    close = 1200 + np.cumsum(np.random.randn(n) * vols)
    df = pd.DataFrame({
        'open':   close - np.abs(np.random.randn(n) * 2),
        'high':   close + np.abs(np.random.randn(n) * 4),
        'low':    close - np.abs(np.random.randn(n) * 4),
        'close':  close,
        'volume': np.random.randint(5000, 30000, n),
    }, index=idx)
    print('Using synthetic regime-switching data')

df = add_returns(df)
df = add_volatility(df)
df = add_momentum(df)
df = add_spread_proxy(df)
df = add_amihud_illiquidity(df)
df_clean = df.dropna()
print(f'{len(df_clean):,} clean bars')

## 2. Volatility Regime Classification

In [ ]:
vr = VolatilityRegime(vol_window=20, low_pct=33, high_pct=67)
df_clean = df_clean.copy()
df_clean['vol_regime'] = vr.fit_predict(df_clean)

regime_counts = df_clean['vol_regime'].value_counts().sort_index()
labels = {0: 'Low Vol', 1: 'Medium Vol', 2: 'High Vol'}
print('Regime counts:')
for k, v in regime_counts.items():
    print(f'  {labels[k]}: {v} bars ({v/len(df_clean)*100:.1f}%)')

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
ax1.plot(df_clean.index, df_clean['close'], lw=1)
for regime_id, color in [(0, 'green'), (1, 'gray'), (2, 'red')]:
    mask = df_clean['vol_regime'] == regime_id
    ax1.fill_between(df_clean.index, df_clean['close'].min(), df_clean['close'].max(),
                     where=mask, alpha=0.15, color=color, label=labels[regime_id])
ax1.set_title('VN30F Price with Volatility Regimes')
ax1.legend(loc='upper left')

ax2.plot(df_clean.index, df_clean['vol_20'], lw=1, label='RV 20-bar')
ax2.set_ylabel('Realised Vol')
ax2.set_title('Realised Volatility')
ax2.legend()
plt.tight_layout()
plt.show()

## 3. HMM Regime Detection

In [ ]:
hmm_features = df_clean[['ret_1', 'vol_20']].dropna()
hmm = HMMRegimeDetector(n_regimes=3, n_iter=300, random_state=42)
df_clean.loc[hmm_features.index, 'hmm_regime'] = hmm.fit_predict(hmm_features)

print('HMM Transition Matrix:')
display(hmm.transition_matrix.round(3))

fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(df_clean.index, df_clean['close'], lw=1, alpha=0.8, label='Close')
for r_id, color in [(0, 'blue'), (1, 'orange'), (2, 'red')]:
    mask = df_clean['hmm_regime'] == r_id
    ax.scatter(df_clean.index[mask], df_clean['close'][mask],
               s=10, alpha=0.4, color=color, label=f'HMM Regime {r_id}')
ax.set_title('VN30F Price — HMM Regime Labels')
ax.legend(loc='upper left', markerscale=2)
plt.tight_layout()
plt.show()

## 4. Walk-Forward Backtest — Mean Reversion Signal

In [ ]:
def mean_rev_signal_func(train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    """Fit Z-score window on train, generate signal on test."""
    # Use training data to calibrate window (here fixed; could be optimised)
    strategy = ZScoreMeanReversion(window=20, entry_z=2.0, exit_z=0.5)
    return strategy.generate(test_df)

wf_results = walk_forward_evaluate(
    df_clean,
    signal_func=mean_rev_signal_func,
    train_size=120,
    test_size=60,
    cost_per_trade=0.0003,   # ~0.03% per trade (VN30F round-trip estimate)
    periods_per_year=252,
)

print('Walk-Forward Results — Mean Reversion:')
display(wf_results.round(3))

print(f'\nMedian OOS Sharpe : {wf_results["sharpe"].median():.3f}')
print(f'Pct folds positive: {(wf_results["sharpe"] > 0).mean()*100:.0f}%')

## 5. Walk-Forward Backtest — Momentum Signal

In [ ]:
def momentum_signal_func(train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    strategy = TimeSeriesMomentum(lookback=20, vol_scale=True)
    return strategy.generate(test_df)

wf_mom = walk_forward_evaluate(
    df_clean,
    signal_func=momentum_signal_func,
    train_size=120,
    test_size=60,
    cost_per_trade=0.0003,
    periods_per_year=252,
)

print('Walk-Forward Results — Momentum:')
display(wf_mom.round(3))

print(f'\nMedian OOS Sharpe : {wf_mom["sharpe"].median():.3f}')
print(f'Pct folds positive: {(wf_mom["sharpe"] > 0).mean()*100:.0f}%')

## 6. Summary Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, results, title in [
    (axes[0], wf_results, 'Mean Reversion'),
    (axes[1], wf_mom, 'Momentum'),
]:
    ax.bar(results['fold'], results['sharpe'], color=[
        'green' if s > 0 else 'red' for s in results['sharpe']
    ])
    ax.axhline(0, color='black', lw=0.8, ls='--')
    ax.set_xlabel('OOS Fold')
    ax.set_ylabel('Sharpe Ratio')
    ax.set_title(f'{title} — OOS Sharpe per Fold')

plt.tight_layout()
plt.show()

print('\n=== Research Summary ===')
print(f'Strategy          | Median Sharpe | % Positive Folds')
print(f'Mean Reversion    | {wf_results["sharpe"].median():>13.3f} | {(wf_results["sharpe"] > 0).mean()*100:>16.0f}%')
print(f'Momentum          | {wf_mom["sharpe"].median():>13.3f} | {(wf_mom["sharpe"] > 0).mean()*100:>16.0f}%')